# **Метрики объяснений: одна картинка, четыре карты, три проверки**

Практика к модулю [«Оценка качества объяснений»](https://ai-interpretability.school/learn/17/0/0).

В модуле мы разобрали несколько способов спросить у карты значимости, говорит ли она правду о модели. Здесь
поставим их рядом на одной картинке и одной модели:

- построим четыре карты: Grad-CAM, Integrated Gradients, SmoothGrad и случайную;
- посчитаем Faithfulness Correlation из библиотеки Quantus;
- построим кривые ROAD — удаление важных пикселей с заполнением по соседям, без переобучения;
- проведем sanity check — рандомизацию весов верхних блоков сети;
- сведем результаты в таблицу «метод × метрика» с рангами и посмотрим, совпадает ли порядок методов.

Автор фото хрюшки: [Brett Sayles, Pexels](https://www.pexels.com/photo/4966539/), лицензия Pexels.

In [ ]:
!pip install -q quantus==0.6.0

In [ ]:
import copy
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import quantus
from io import BytesIO
from PIL import Image
from scipy.stats import spearmanr
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
from quantus.functions.perturb_func import noisy_linear_imputation

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('устройство:', device)

## Модель и изображение

ResNet-50 с весами ImageNet и фотография из открытого репозитория курса — та же связка, что в практиках модулей
«Эвристические объяснения на основе градиента» и «Объяснения на основе локализации и CAM». Режим `eval()` обязателен: в режиме обучения BatchNorm считает статистику по
батчу из одной картинки, и прогноз перестает совпадать с настоящим.

In [ ]:
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).to(device).eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
image = Image.open(BytesIO(requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/piglet.jpg').content)).convert('RGB')
x = transform(image).unsqueeze(0).to(device)
categories = requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/imagenet_classes.txt').text.splitlines()

with torch.no_grad():
    probs = F.softmax(model(x), dim=1)[0]
target = probs.argmax().item()
print(f'класс {target} «{categories[target]}», вероятность {probs[target].item():.3f}')

## 1. Четыре карты

Grad-CAM и SmoothGrad даны готовыми: тетради нужны их карты, а не устройство методов. Случайная
карта от модели не зависит вовсе: это точка сравнения, хуже которой метод быть не должен. Весь случайный шум
генерируется на CPU с фиксированным сидом, поэтому карты одинаковы на CPU и на GPU.

SmoothGrad здесь усредняет модули градиентов по зашумленным копиям, как `captum.attr.NoiseTunnel` поверх `Saliency` с `abs=True`. У Smilkov et al. (2017) усредняют сами градиенты, а модуль берут уже для картинки; это две разные карты.

In [ ]:
def gradient(model, x, target):
    """Градиент логита класса по входу."""
    x = x.clone().requires_grad_(True)
    grad, = torch.autograd.grad(model(x)[0, target], x)
    return grad

def grad_cam(model, x, target):
    """Grad-CAM по выходу layer4, растянутый до размера картинки."""
    store = {}
    hook = model.layer4.register_forward_hook(lambda module, inp, out: store.update(act=out))
    logit = model(x)[0, target]
    hook.remove()
    grad, = torch.autograd.grad(logit, store['act'])
    cam = F.relu((grad.mean(dim=(2, 3), keepdim=True) * store['act']).sum(1, keepdim=True))
    return F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)[0, 0].detach()

def smoothgrad(model, x, target, sigma_frac=0.15, n=25):
    """SmoothGrad: средняя карта |градиента| по n зашумленным копиям входа."""
    gen = torch.Generator().manual_seed(0)
    sigma = sigma_frac * (x.max() - x.min())
    acc = torch.zeros(x.shape[-2:], device=x.device)
    for _ in range(n):
        noise = torch.randn(x.shape, generator=gen).to(x.device)
        acc += gradient(model, x + noise * sigma, target).abs().max(dim=1).values[0]
    return (acc / n).detach()

def random_map(model, x, target):
    """Случайная карта: от модели не зависит."""
    return torch.rand(x.shape[-2:], generator=torch.Generator().manual_seed(0)).to(x.device)

**Задание 1.** Напишите Integrated Gradients с нулевым эталонным входом (после нормализации ноль — это
средний цвет ImageNet) и $m$ шагами по пути: среднее градиентов в точках $\alpha x$, $\alpha = 1/m, 2/m, \dots, 1$,
умноженное на $x$ и просуммированное по каналам. Проверьте для IG аксиому полноты (completeness): сумма карты должна быть близка к
разности логитов $f(x) - f(0)$. Посчитайте сумму карты при $m = 32$ и при $m = 128$ и сравните с разностью
логитов. Дальше в тетради используется $m = 128$. Здесь интеграл считается суммой Римана по правым концам, как у Sundararajan et al. (2017); `captum.attr.IntegratedGradients` по умолчанию берет узлы Гаусса — Лежандра (`method='gausslegendre'`, `n_steps=50`), и сумма карты у него на тех же $m$ выходит другой.

In [ ]:
# Ваш код здесь

**Ваш ответ:**

In [ ]:
methods = {'Grad-CAM': grad_cam, 'IG': integrated_gradients, 'SmoothGrad': smoothgrad, 'Random': random_map}
maps = {name: method(model, x, target) for name, method in methods.items()}

shown = transforms.Resize((224, 224))(image)
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(shown)
axes[0].set_title(f'Вход: {categories[target]}')
for ax, (name, m) in zip(axes[1:], maps.items()):
    ax.imshow(m.abs().cpu().numpy(), cmap='hot')
    ax.set_title(name)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Faithfulness Correlation в Quantus

Метрика из работы [Bhatt et al., 2020](https://arxiv.org/abs/2005.00631): выбирается случайное подмножество
пикселей $S$, его значения заменяются эталоном, и считаются две величины — сумма значимости пикселей
подмножества $\sum_{i \in S} a_i$ и падение выхода модели $f(x) - f(x_{S \leftarrow \text{baseline}})$. Повторив это
много раз, берут корреляцию Пирсона между двумя величинами. Чем она выше, тем лучше карта предсказывает,
какое подмножество сильнее повлияет на ответ.

В Quantus это `quantus.FaithfulnessCorrelation`; по умолчанию $|S| = 224$ пикселя из $224 \times 224$ и 100
повторов. Подмножества выбираются через `np.random`, поэтому результат зависит от сида.

**Задание 2.** Посчитайте Faithfulness Correlation для четырех карт с настройками по умолчанию (черный
эталон, 100 повторов, $|S| = 224$) при сидах `np.random.seed(0)` … `np.random.seed(4)`. Выведите для каждой карты
среднее и стандартное отклонение по пяти сидам. Отличается ли разница между картами от разброса между сидами?

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 3. ROAD: удаление важных пикселей с заполнением по соседям

ROAD ([Rong et al., 2022](https://arxiv.org/abs/2202.00449)) удаляет долю пикселей по порядку карты и заполняет
удаленные пиксели функцией `noisy_linear_imputation` из Quantus: каждый удаленный пиксель получает взвешенное среднее соседей
и небольшой шум. Модель не переобучается. Удалять можно в двух порядках:

- MoRF (most relevant first) — сначала самые важные пиксели; хорошая карта быстро роняет вероятность класса,
  чем ниже средняя вероятность по кривой, тем лучше;
- LeRF (least relevant first) — сначала наименее важные; хорошая карта долго держит вероятность класса, чем выше
  средняя вероятность по кривой, тем лучше.

Кривая — вероятность класса при удалении 5%, 10%, …, 95% пикселей; итог кривой — среднее по 19 точкам.

In [ ]:
fractions = np.arange(5, 100, 5) / 100

def road_curve(m, order):
    """Вероятность класса после удаления доли пикселей в порядке MoRF или LeRF."""
    importance = m.cpu().numpy().flatten()
    ranked = np.argsort(-importance) if order == 'MoRF' else np.argsort(importance)
    np.random.seed(0)                                   # шум заполнения воспроизводим
    curve = []
    for frac in fractions:
        filled = noisy_linear_imputation(x_np[0].copy(), ranked[:int(frac * importance.size)], noise=0.01)
        with torch.no_grad():
            out = F.softmax(model(torch.tensor(filled[None], dtype=torch.float32, device=device)), dim=1)
        curve.append(out[0, target].item())
    return np.array(curve)

**Задание 3.** Постройте кривые MoRF и LeRF для четырех карт и нарисуйте их. Посчитайте среднюю вероятность
класса по каждой кривой. Какая карта лучшая по MoRF и какая по LeRF? Совпадает ли порядок четырех карт по MoRF и
по LeRF?

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 4. Sanity check: рандомизация весов

Проверка из работы [Adebayo et al., 2018](https://arxiv.org/abs/1810.03292): веса сети накопительно
переинициализируются от выхода ко входу — сначала `fc`, потом `fc` и `layer4` и так далее, — и карта строится
заново. Мера сходства — корреляция Спирмена между исходной картой и новой, вытянутыми в вектор. Карта, которая
объясняет модель, обязана измениться: чем ближе корреляция к нулю, тем лучше, поэтому сравнивать карты будем по
модулю корреляции. Веса переинициализируются на CPU с
фиксированным сидом, поэтому сломанная сеть одна и та же на CPU и на GPU.

In [ ]:
def randomize(model, blocks, seed=0):
    """Копия сети со случайно переинициализированными весами перечисленных блоков."""
    broken = copy.deepcopy(model).cpu()
    torch.manual_seed(seed)
    for block in blocks:
        for module in getattr(broken, block).modules():
            if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear, torch.nn.BatchNorm2d)):
                module.reset_parameters()
    return broken.to(device).eval()

cascade = [['fc'], ['fc', 'layer4'], ['fc', 'layer4', 'layer3'], ['fc', 'layer4', 'layer3', 'layer2']]

**Задание 4.** Для каждого шага каскада и каждой из четырех карт посчитайте корреляцию Спирмена между
картой обученной сети и картой сломанной сети. Чему равна корреляция для SmoothGrad после переинициализации
только `fc` (округлите до сотых)? Почему у случайной карты корреляция равна 1 на всех шагах каскада и что происходит с
Grad-CAM на последнем шаге каскада?

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 5. Таблица «метод × метрика»

**Задание 5.** Сведите в одну таблицу три метрики: среднее Faithfulness Correlation по пяти сидам (больше —
лучше), среднюю вероятность класса по кривой ROAD MoRF (меньше — лучше) и модуль корреляции Спирмена после
переинициализации `fc` и `layer4` (меньше — лучше). Поставьте каждой карте ранг по каждой метрике (1 — лучшая).
Совпадает ли порядок карт по трем метрикам?

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## Что унести из тетради

- **Порядок методов зависит от метрики.** На одной картинке и одной модели Grad-CAM — лучшая из четырех карт по
  ROAD MoRF, последняя по ROAD LeRF и третья по sanity check.
- **Случайная карта — обязательная точка сравнения.** Только она показывает, отличает ли метрика метод от
  угадывания: Faithfulness Correlation с настройками по умолчанию поставила случайную карту выше двух методов.
- **Одна картинка — это шум.** Разброс Faithfulness Correlation между сидами того же порядка, что разница между
  картами, а кривые ROAD немонотонны. Выводы о методе делают по набору картинок и нескольким сидам.
- **Sanity check запускают каскадом.** SmoothGrad пережил переинициализацию `fc` с корреляцией 0.77 и
  изменился только после `layer4` — это видно только на отдельных шагах каскада.